In [1]:
#!/usr/bin/env python3

"""
Create multi-year spatial co-exposure rasters for compound disturbances.

This analysis uses SPATIAL logic within each temporal window.

For each disturbance, a pixel is first classified according to whether that
disturbance occurred at least once anywhere within the period.

Compound disturbance is then calculated from those period-level presence
rasters.

Example
-------
If a pixel experiences:

    wildfire in 2001
    biotic disturbance in 2003
    hotter drought in 2004

then for the 2000-2004 period:

    wf_bt       = 1
    wf_hd       = 1
    bt_hd       = 1
    wf_bt_hd    = 1

The disturbances DO NOT need to occur in the same year.

This differs from the annual co-occurrence/frequency analysis, which requires
the disturbances to occur simultaneously within a given year.

Periods
-------
    2000-2004
    2005-2009
    2010-2014
    2015-2020

The first three periods contain 5 years and the final period contains 6 years.

Inputs
------
Annual ANY binary stacks:

    data/derived/annual_stacks/binary/
        annual_stack_any_YYYY.tif

Band order:

    1 = wildfire
    2 = biotic disturbance
    3 = hotter drought
    4 = PDSI

Only bands 1-3 are used.

Outputs
-------
Binary 30 m rasters:

    0 = combination absent during the period
    1 = combination present during the period
    255 = NoData / outside forest

Outputs are written to:

    data/derived/temporal_spatial_coexposure/

All calculations are performed block-by-block to minimize memory use.
"""


from pathlib import Path

import numpy as np
import rasterio


# =============================================================================
# Paths
# =============================================================================

PROJECT_DIR = Path(
    "/home/jovyan/data-store/forest-disturbance-stack-v3"
)

ANNUAL_DIR = (
    PROJECT_DIR
    / "data"
    / "derived"
    / "annual_stacks"
    / "binary"
)

FOREST_MASK = (
    PROJECT_DIR
    / "data"
    / "derived"
    / "resampled"
    / "forest_mask_30m_resampled.tif"
)

OUTPUT_DIR = (
    PROJECT_DIR
    / "data"
    / "derived"
    / "temporal_spatial_coexposure"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# =============================================================================
# Temporal periods
#
# The first three periods contain 5 years.
# The final period contains 6 years so that the complete 2000-2020 record
# is represented.
# =============================================================================

PERIODS = {

    "2000_2004": range(
        2000,
        2005
    ),

    "2005_2009": range(
        2005,
        2010
    ),

    "2010_2014": range(
        2010,
        2015
    ),

    "2015_2020": range(
        2015,
        2021
    ),
}


# =============================================================================
# Annual stack band numbers
#
# rasterio uses 1-based band indexing.
# =============================================================================

WF_BAND = 1
BT_BAND = 2
HD_BAND = 3


# =============================================================================
# Compound disturbances
# =============================================================================

COMPOUNDS = [
    "wf_bt",
    "wf_hd",
    "bt_hd",
    "wf_bt_hd",
]


# =============================================================================
# Output settings
#
# Outputs are binary Byte rasters:
#
#   0   = combination absent
#   1   = combination present
#   255 = NoData / outside forest
# =============================================================================

OUTPUT_DTYPE = "uint8"

OUTPUT_NODATA = 255


# =============================================================================
# Helper: annual raster path
# =============================================================================

def annual_path(year):

    return (
        ANNUAL_DIR
        / f"annual_stack_any_{year}.tif"
    )


# =============================================================================
# Validate input files
# =============================================================================

print("\nChecking input files...\n")


# Full 2000-2020 record
all_years = range(
    2000,
    2021
)


for year in all_years:

    path = annual_path(year)

    if not path.exists():

        raise FileNotFoundError(
            f"Missing annual stack: {path}"
        )


if not FOREST_MASK.exists():

    raise FileNotFoundError(
        f"Missing forest mask: {FOREST_MASK}"
    )


print("All required annual stacks found.")
print("Forest mask found.")


# =============================================================================
# Check raster alignment
# =============================================================================

print("\nChecking raster alignment...\n")


reference_path = annual_path(2000)


with rasterio.open(reference_path) as ref:

    ref_width = ref.width
    ref_height = ref.height
    ref_transform = ref.transform
    ref_crs = ref.crs


# Check annual stacks
for year in all_years:

    path = annual_path(year)

    with rasterio.open(path) as src:

        if src.width != ref_width:

            raise ValueError(
                f"Width mismatch: {path}"
            )

        if src.height != ref_height:

            raise ValueError(
                f"Height mismatch: {path}"
            )

        if src.transform != ref_transform:

            raise ValueError(
                f"Transform mismatch: {path}"
            )

        if src.crs != ref_crs:

            raise ValueError(
                f"CRS mismatch: {path}"
            )


# Check forest mask
with rasterio.open(FOREST_MASK) as forest_src:

    if forest_src.width != ref_width:

        raise ValueError(
            "Forest mask width does not match annual stacks."
        )

    if forest_src.height != ref_height:

        raise ValueError(
            "Forest mask height does not match annual stacks."
        )

    if forest_src.transform != ref_transform:

        raise ValueError(
            "Forest mask transform does not match annual stacks."
        )

    if forest_src.crs != ref_crs:

        raise ValueError(
            "Forest mask CRS does not match annual stacks."
        )


print("All rasters are aligned.")


# =============================================================================
# Process one temporal period
# =============================================================================

def process_period(
    period_name,
    years
):

    years = list(years)

    print("\n" + "=" * 70)

    print(
        f"Processing period: {period_name}"
    )

    print(
        f"Years: {years}"
    )

    print("=" * 70)


    # -------------------------------------------------------------------------
    # Open annual stacks
    # -------------------------------------------------------------------------

    annual_sources = [

        rasterio.open(
            annual_path(year)
        )

        for year in years
    ]


    forest_src = rasterio.open(
        FOREST_MASK
    )


    try:

        # ---------------------------------------------------------------------
        # Output raster profile
        # ---------------------------------------------------------------------

        profile = (
            annual_sources[0]
            .profile
            .copy()
        )


        profile.update(
            count=1,
            dtype=OUTPUT_DTYPE,
            nodata=OUTPUT_NODATA,
            compress="deflate",
            predictor=2,
            tiled=True,
            BIGTIFF="YES",
        )


        # ---------------------------------------------------------------------
        # Output paths
        # ---------------------------------------------------------------------

        output_paths = {

            compound: (
                OUTPUT_DIR
                / (
                    f"{compound}_"
                    f"{period_name}_"
                    f"presence.tif"
                )
            )

            for compound in COMPOUNDS
        }


        # ---------------------------------------------------------------------
        # Open output rasters
        # ---------------------------------------------------------------------

        outputs = {

            compound: rasterio.open(
                output_paths[compound],
                "w",
                **profile
            )

            for compound in COMPOUNDS
        }


        try:

            # -----------------------------------------------------------------
            # Process block-by-block
            # -----------------------------------------------------------------

            windows = list(
                annual_sources[0]
                .block_windows(1)
            )

            n_windows = len(windows)


            for window_number, (_, window) in enumerate(
                windows,
                start=1
            ):

                # -------------------------------------------------------------
                # Forest mask
                # -------------------------------------------------------------

                forest = forest_src.read(
                    1,
                    window=window
                )

                forest_bool = (
                    forest > 0
                )


                # -------------------------------------------------------------
                # Initialize period-level presence arrays
                #
                # These answer:
                #
                # "Did this disturbance occur at least once during this
                #  temporal period?"
                # -------------------------------------------------------------

                shape = (
                    int(window.height),
                    int(window.width)
                )


                wf_period = np.zeros(
                    shape,
                    dtype=bool
                )

                bt_period = np.zeros(
                    shape,
                    dtype=bool
                )

                hd_period = np.zeros(
                    shape,
                    dtype=bool
                )


                # -------------------------------------------------------------
                # Collapse annual presence across the period
                #
                # Logical OR is used independently for each disturbance.
                # -------------------------------------------------------------

                for src in annual_sources:

                    wf = (
                        src.read(
                            WF_BAND,
                            window=window
                        )
                        > 0
                    )

                    bt = (
                        src.read(
                            BT_BAND,
                            window=window
                        )
                        > 0
                    )

                    hd = (
                        src.read(
                            HD_BAND,
                            window=window
                        )
                        > 0
                    )


                    wf_period |= wf
                    bt_period |= bt
                    hd_period |= hd


                # -------------------------------------------------------------
                # Calculate compound spatial co-exposure
                #
                # Intersections occur AFTER each disturbance has been
                # collapsed across the complete temporal period.
                #
                # Therefore, disturbances do not need to occur in the same
                # year to be considered spatially co-exposed.
                # -------------------------------------------------------------

                compounds = {

                    "wf_bt":
                        wf_period
                        & bt_period,

                    "wf_hd":
                        wf_period
                        & hd_period,

                    "bt_hd":
                        bt_period
                        & hd_period,

                    "wf_bt_hd":
                        wf_period
                        & bt_period
                        & hd_period,
                }


                # -------------------------------------------------------------
                # Write each compound raster
                # -------------------------------------------------------------

                for compound in COMPOUNDS:

                    out = (
                        compounds[compound]
                        .astype(np.uint8)
                    )


                    # ---------------------------------------------------------
                    # Outside forest = NoData
                    #
                    # Within forest:
                    #
                    #   0 = combination absent
                    #   1 = combination present
                    # ---------------------------------------------------------

                    out[
                        ~forest_bool
                    ] = OUTPUT_NODATA


                    outputs[
                        compound
                    ].write(
                        out,
                        1,
                        window=window
                    )


                # -------------------------------------------------------------
                # Progress
                # -------------------------------------------------------------

                if (
                    window_number % 100 == 0
                    or
                    window_number == n_windows
                ):

                    print(
                        f"  Block "
                        f"{window_number:,} / "
                        f"{n_windows:,}"
                    )


        finally:

            for dst in outputs.values():

                dst.close()


        # ---------------------------------------------------------------------
        # Report outputs
        # ---------------------------------------------------------------------

        print("\nCreated:")


        for compound in COMPOUNDS:

            print(
                f"  "
                f"{output_paths[compound]}"
            )


    finally:

        for src in annual_sources:

            src.close()


        forest_src.close()


# =============================================================================
# Run all four periods
# =============================================================================

for (
    period_name,
    period_years
) in PERIODS.items():

    process_period(
        period_name,
        period_years
    )


print("\n" + "=" * 70)

print(
    "Temporal spatial co-exposure rasters complete."
)

print("=" * 70)


Checking input files...

All required annual stacks found.
Forest mask found.

Checking raster alignment...

All rasters are aligned.

Processing period: 2000_2004
Years: [2000, 2001, 2002, 2003, 2004]
  Block 100 / 68,970
  Block 200 / 68,970
  Block 300 / 68,970
  Block 400 / 68,970
  Block 500 / 68,970
  Block 600 / 68,970
  Block 700 / 68,970
  Block 800 / 68,970
  Block 900 / 68,970
  Block 1,000 / 68,970
  Block 1,100 / 68,970
  Block 1,200 / 68,970
  Block 1,300 / 68,970
  Block 1,400 / 68,970
  Block 1,500 / 68,970
  Block 1,600 / 68,970
  Block 1,700 / 68,970
  Block 1,800 / 68,970
  Block 1,900 / 68,970
  Block 2,000 / 68,970
  Block 2,100 / 68,970
  Block 2,200 / 68,970
  Block 2,300 / 68,970
  Block 2,400 / 68,970
  Block 2,500 / 68,970
  Block 2,600 / 68,970
  Block 2,700 / 68,970
  Block 2,800 / 68,970
  Block 2,900 / 68,970
  Block 3,000 / 68,970
  Block 3,100 / 68,970
  Block 3,200 / 68,970
  Block 3,300 / 68,970
  Block 3,400 / 68,970
  Block 3,500 / 68,970
  Block 3,